In [1]:
from dotenv import load_dotenv
import os

from src.models.llm_model import OpenrouterLLMModel
load_dotenv("..")


import pandas as pd
from tqdm import tqdm
from functools import partial
import joblib
import numpy as np
import faiss
import json
from sentence_transformers import SentenceTransformer
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder


from src.data.rac_utils import retrive_related

c:\Work\Project\ticket-nlp-classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_PATH = "../data/raw/all_tickets_processed_improved_v3.csv"
df = pd.read_csv(DATA_PATH)

X = df["Document"]
y = df["Topic_group"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, shuffle=True, random_state=2)

vectorizer: TfidfVectorizer = joblib.load("../artifacts/tfidf_vectorizer_v01.pkl")

with open("../artifacts/rac_corpus_similarity-euclidian_index_v01.json", 'r') as f:
    corpus = json.load(f)

index = faiss.read_index("../artifacts/traindata_similarity_index_v01.index")

retrieval_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

labelencoder: LabelEncoder = joblib.load("../artifacts/labelencoder_neural_v01.pkl")

#### Zero shot

In [3]:
model = OpenrouterLLMModel(
    API_KEY=os.environ.get("OPENROUTER_KEY"),
    MODEL="meta-llama/llama-3.3-70b-instruct:free",
    base_url="https://openrouter.ai/api/v1/chat/completions"
)

In [4]:
from typing import Literal
from pydantic import BaseModel, ConfigDict, Field

class TestD(BaseModel):
    model_config = ConfigDict(strict=True, extra="forbid")
    ticket_class: Literal[
        'Access',
        'Administrative rights',
        'HR Support',
        'Hardware',
        'Internal Project',
        'Miscellaneous',
        'Purchase',
        'Storage'
    ] = Field(..., description="Select a appropriate class for the given ticket")
    
    
    
system_prompt = """
You are AI assistant specializing in classifying IT tickets based on ticket description.

Task:
- Details of the ticket will be given in the 'Ticket description' section.
- Your task is to classify them into proper classes.
- Output should be in a json parsable format.
"""

user_prompt = """
Ticket description: {context}

####

Output:
"""

In [5]:
model.set_system_prompt(system_prompt)
model.set_schema(TestD)

In [16]:
# test with 50
import time

X_test_list = X_test.to_list()
response_class = {}
for i in range(50):
    response_data = model.invoke(
        user_prompt=user_prompt.format(
            context = X_test_list[i]
            ),
        temperature = 0
    )

    try:
        _d = TestD.model_validate_json(response_data["choices"][0]["message"]["content"])
        response_class[i] = _d.ticket_class
    except Exception as e:
        print("Parsing error for example" + str(i))
        response_class[i] = "ERROR"
    
    time.sleep(1.1)

Parsing error for example4
Parsing error for example5
Parsing error for example8


HTTPError: 429 Client Error: Too Many Requests for url: https://openrouter.ai/api/v1/chat/completions

In [17]:
response_class

{0: 'Hardware',
 1: 'Access',
 2: 'Hardware',
 3: 'HR Support',
 4: 'ERROR',
 5: 'ERROR',
 6: 'Access',
 7: 'Hardware',
 8: 'ERROR'}

In [14]:
len(y_test.to_list()[0:30])
# len(list(response_class.values()))

30

In [15]:
from src.evaluation.metrics import evaluate, format_cm
format_cm(evaluate(y_test.to_list()[0:30], list(response_class.values())), normalize=True)

               precision    recall  f1-score   support

       Access     0.3750    0.7500    0.5000         4
   HR Support     0.7143    0.7143    0.7143         7
     Hardware     0.5385    0.7000    0.6087        10
Miscellaneous     0.0000    0.0000    0.0000         4
     Purchase     0.0000    0.0000    0.0000         3
      Storage     0.0000    0.0000    0.0000         2

     accuracy                         0.5000        30
    macro avg     0.2713    0.3607    0.3038        30
 weighted avg     0.3962    0.5000    0.4362        30



c:\Work\Project\ticket-nlp-classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Work\Project\ticket-nlp-classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Work\Project\ticket-nlp-classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{me

,Pred: class-0,Pred: class-1,Pred: class-2,Pred: class-3,Pred: class-4,Pred: class-5
True: class-0,0.750000,0.000000,0.25,0.0,0.0,0.0
True: class-1,0.285714,0.714286,0.00,0.0,0.0,0.0
True: class-2,0.100000,0.000000,0.70,0.1,0.0,0.1
True: class-3,0.250000,0.500000,0.25,0.0,0.0,0.0
True: class-4,0.000000,0.000000,1.00,0.0,0.0,0.0
True: class-5,0.500000,0.000000,0.50,0.0,0.0,0.0
